# ⚡ PySpark Complete Tutorial & Last-Minute Cheat Sheet

Welcome to the **PySpark Complete Hands-On Tutorial & Quick Reference Cheat Sheet**! 🚀

PySpark is the Python API for **Apache Spark**, an open-source, distributed computing framework designed for fast processing of large-scale data (Big Data Analytics & Machine Learning).

---

## 📌 Table of Contents
1. [PySpark Initialization & SparkSession](#1-pyspark-initialization--sparksession)
2. [Creating DataFrames & Custom Schemas](#2-creating-dataframes--custom-schemas)
3. [Data Ingestion & Dataset Inspection](#3-data-ingestion--dataset-inspection)
4. [Column Selection & Expression Aliasing](#4-column-selection--expression-aliasing)
5. [Data Filtering & Conditional Logic](#5-data-filtering--conditional-logic)
6. [Adding, Modifying & Dropping Columns](#6-adding-modifying--dropping-columns)
7. [Data Sorting & Ordering](#7-data-sorting--ordering)
8. [Deduplication & Missing Value Handling](#8-deduplication--missing-value-handling)
9. [GroupBy & Aggregation Functions](#9-groupby--aggregation-functions)
10. [DataFrame Joins](#10-dataframe-joins)
11. [Spark SQL Engine Execution](#11-spark-sql-engine-execution)
12. [User Defined Functions (UDFs)](#12-user-defined-functions-udfs)
13. [Date & Timestamp Manipulations](#13-date--timestamp-manipulations)
14. [String Functions](#14-string-functions)
15. [Advanced Window Functions](#15-advanced-window-functions)
16. [Pandas Interoperability & Data Exporting](#16-pandas-interoperability--data-exporting)
17. [PySpark Quick Reference Cheat Sheet](#17-pyspark-quick-reference-cheat-sheet)

---

In [ ]:
# Install PySpark (Uncomment if running in Google Colab or fresh environment)
# !pip install pyspark -q

## 1. PySpark Initialization & SparkSession

A `SparkSession` is the entry point to PySpark applications. It manages connection to the Spark cluster and allows creating DataFrames, registering UDFs, and executing SQL queries.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("PySpark_Tutorial_CheatSheet") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")

Spark Version: 4.2.0
Application Name: PySpark_Tutorial_CheatSheet


## 2. Creating DataFrames & Custom Schemas

PySpark DataFrames can be created manually from Python lists/tuples or using explicit schemas defined via `StructType` and `StructField`.

In [ ]:
# Method 1: Inferring schema from Python tuples
data = [
    (1, "Swapnanil", 23, "Data Engineer", 85000.0),
    (2, "Debraj", 22, "Data Analyst", 72000.0),
    (3, "Swagata", 23, "ML Engineer", 90000.0)
]
columns = ["id", "name", "age", "role", "salary"]

df_manual = spark.createDataFrame(data, columns)
df_manual.show()

+---+---------+---+-------------+-------+
| id|     name|age|         role| salary|
+---+---------+---+-------------+-------+
|  1|Swapnanil| 23|Data Engineer|85000.0|
|  2|   Debraj| 22| Data Analyst|72000.0|
|  3|  Swagata| 23|  ML Engineer|90000.0|
+---+---------+---+-------------+-------+



In [ ]:
# Method 2: Creating DataFrame with explicit StructType Schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("role", StringType(), True),
    StructField("salary", DoubleType(), True)
])

df_schema = spark.createDataFrame(data, schema=schema)
df_schema.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- role: string (nullable = true)
 |-- salary: double (nullable = true)



## 3. Data Ingestion & Dataset Inspection

In real-world data pipelines, PySpark reads datasets from various storage layers (CSV, Parquet, JSON, databases). Here we load `popularity_campaign_v2 1.csv`.

In [ ]:
# Load CSV file with headers and schema inference
csv_file_path = "popularity_campaign_v2 1.csv"
df = spark.read.csv(csv_file_path, header=True, inferSchema=True)

# Preview first 5 rows
df.show(5, truncate=False)

+-------+----------+------------------------+-------+-----+-----------+------------+------------+-----------+
|user_id|content_id|user_specialty          |item_id|score|filter_used|channel_code|content_type|template_id|
+-------+----------+------------------------+-------+-----+-----------+------------+------------+-----------+
|131060 |36464     |Obstetrics & Gynaecology|36464_1|0.01 |personalize|sms         |1           |6038.0     |
|131064 |0         |Paediatrics             |0_2    |0.01 |personalize|sms         |2           |5679.0     |
|131065 |11350     |Internal Medicine       |11350_2|0.01 |personalize|sms         |2           |5679.0     |
|131077 |0         |Paediatrics             |0_2    |0.01 |personalize|sms         |2           |5679.0     |
|131098 |0         |Paediatrics             |0_2    |0.01 |personalize|sms         |2           |5679.0     |
+-------+----------+------------------------+-------+-----+-----------+------------+------------+-----------+
only showi

In [ ]:
# Inspecting DataFrame properties
print("Schema overview:")
df.printSchema()

print(f"Total Rows: {df.count()}")
print(f"Columns: {df.columns}")

Schema overview:
root
 |-- user_id: integer (nullable = true)
 |-- content_id: integer (nullable = true)
 |-- user_specialty: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- score: double (nullable = true)
 |-- filter_used: string (nullable = true)
 |-- channel_code: string (nullable = true)
 |-- content_type: integer (nullable = true)
 |-- template_id: double (nullable = true)

Total Rows: 9507
Columns: ['user_id', 'content_id', 'user_specialty', 'item_id', 'score', 'filter_used', 'channel_code', 'content_type', 'template_id']


In [ ]:
# Statistical summary of numerical columns
df.describe().show()

+-------+-----------------+------------------+--------------------+-------+--------------------+-----------+------------+------------------+-----------------+
|summary|          user_id|        content_id|      user_specialty|item_id|               score|filter_used|channel_code|      content_type|      template_id|
+-------+-----------------+------------------+--------------------+-------+--------------------+-----------+------------+------------------+-----------------+
|  count|             9507|              9507|                9507|   9507|                9507|       9507|        9507|              9507|             9507|
|   mean|397270.7656463658| 26200.43452193121|                NULL|   NULL|0.010000000000001234|       NULL|        NULL|1.3496371094982644|6826.336383717261|
| stddev|160431.1193273139|15944.713546157127|                NULL|   NULL|                 0.0|       NULL|        NULL|0.5505965481256534| 1726.72944185038|
|    min|           131060|                 0|

## 4. Column Selection & Expression Aliasing

Use `.select()` to project specific columns and `col().alias()` or `selectExpr()` for expressions and renaming.

In [ ]:
# Select specific columns
df.select("user_id", "user_specialty", "score").show(5)

# Select with alias using col()
df.select(
    col("user_id").alias("doctor_id"),
    col("user_specialty").alias("specialization"),
    col("score")
).show(5)

# Select using SQL expressions via selectExpr()
df.selectExpr("user_id", "score * 100 as total_score_pct", "upper(channel_code) as channel").show(5)

+-------+--------------------+-----+
|user_id|      user_specialty|score|
+-------+--------------------+-----+
| 131060|Obstetrics & Gyna...| 0.01|
| 131064|         Paediatrics| 0.01|
| 131065|   Internal Medicine| 0.01|
| 131077|         Paediatrics| 0.01|
| 131098|         Paediatrics| 0.01|
+-------+--------------------+-----+
only showing top 5 rows
+---------+--------------------+-----+
|doctor_id|      specialization|score|
+---------+--------------------+-----+
|   131060|Obstetrics & Gyna...| 0.01|
|   131064|         Paediatrics| 0.01|
|   131065|   Internal Medicine| 0.01|
|   131077|         Paediatrics| 0.01|
|   131098|         Paediatrics| 0.01|
+---------+--------------------+-----+
only showing top 5 rows
+-------+---------------+-------+
|user_id|total_score_pct|channel|
+-------+---------------+-------+
| 131060|            1.0|    SMS|
| 131064|            1.0|    SMS|
| 131065|            1.0|    SMS|
| 131077|            1.0|    SMS|
| 131098|            1.0|    S

## 5. Data Filtering & Conditional Logic

Filter dataset rows based on single or multiple criteria using `.filter()` or `.where()`.

In [ ]:
# Filter 1: Score threshold condition
df.filter(col("score") > 0.005).show(5)

# Filter 2: Multiple conditions using AND (&)
df.filter((col("score") > 0.005) & (col("user_specialty") == "Internal Medicine")).show(5)

# Filter 3: Multiple conditions using OR (|)
df.filter((col("score") > 0.01) | (col("user_specialty") == "Internal Medicine")).show(5)

# Filter 4: Set membership using .isin()
df.filter(col("user_specialty").isin("Paediatrics", "ENT", "Orthopaedics")).show(5)

# Filter 5: Inequality filter
df.filter(col("user_specialty") != "Internal Medicine").show(5)

+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
|user_id|content_id|      user_specialty|item_id|score|filter_used|channel_code|content_type|template_id|
+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
| 131060|     36464|Obstetrics & Gyna...|36464_1| 0.01|personalize|         sms|           1|     6038.0|
| 131064|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131065|     11350|   Internal Medicine|11350_2| 0.01|personalize|         sms|           2|     5679.0|
| 131077|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131098|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
only showing top 5 rows
+-------+----------+--

## 6. Adding, Modifying & Dropping Columns

Use `.withColumn()` to add new columns or replace existing ones, and `.drop()` to remove unnecessary columns.

In [ ]:
# Add a static literal column
df_transformed = df.withColumn("Country", lit("India"))

# Add a calculated column
df_transformed = df_transformed.withColumn("final_score", col("score") * 100)

# Add a conditional column using when().otherwise()
df_transformed = df_transformed.withColumn(
    "score_category",
    when(col("score") > 0.005, "High").otherwise("Low")
)

# Rename a column
df_transformed = df_transformed.withColumnRenamed("channel_code", "communication_channel")

df_transformed.show(5)

+-------+----------+--------------------+-------+-----+-----------+---------------------+------------+-----------+-------+-----------+--------------+
|user_id|content_id|      user_specialty|item_id|score|filter_used|communication_channel|content_type|template_id|Country|final_score|score_category|
+-------+----------+--------------------+-------+-----+-----------+---------------------+------------+-----------+-------+-----------+--------------+
| 131060|     36464|Obstetrics & Gyna...|36464_1| 0.01|personalize|                  sms|           1|     6038.0|  India|        1.0|          High|
| 131064|         0|         Paediatrics|    0_2| 0.01|personalize|                  sms|           2|     5679.0|  India|        1.0|          High|
| 131065|     11350|   Internal Medicine|11350_2| 0.01|personalize|                  sms|           2|     5679.0|  India|        1.0|          High|
| 131077|         0|         Paediatrics|    0_2| 0.01|personalize|                  sms|           

In [ ]:
# Drop columns
df_dropped = df_transformed.drop("filter_used", "template_id")
df_dropped.show(5)

+-------+----------+--------------------+-------+-----+---------------------+------------+-------+-----------+--------------+
|user_id|content_id|      user_specialty|item_id|score|communication_channel|content_type|Country|final_score|score_category|
+-------+----------+--------------------+-------+-----+---------------------+------------+-------+-----------+--------------+
| 131060|     36464|Obstetrics & Gyna...|36464_1| 0.01|                  sms|           1|  India|        1.0|          High|
| 131064|         0|         Paediatrics|    0_2| 0.01|                  sms|           2|  India|        1.0|          High|
| 131065|     11350|   Internal Medicine|11350_2| 0.01|                  sms|           2|  India|        1.0|          High|
| 131077|         0|         Paediatrics|    0_2| 0.01|                  sms|           2|  India|        1.0|          High|
| 131098|         0|         Paediatrics|    0_2| 0.01|                  sms|           2|  India|        1.0|        

## 7. Data Sorting & Ordering

Sort rows using `.orderBy()` or `.sort()` with `asc()` or `desc()`.

In [ ]:
# Ascending sort by content_id
df.orderBy("content_id").show(5)

# Descending sort by content_id
df.orderBy(col("content_id").desc()).show(5)

# Multi-column sort (user_specialty ascending, content_id descending)
df.orderBy(col("user_specialty").asc(), col("content_id").desc()).show(5)

+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
|user_id|content_id|      user_specialty|item_id|score|filter_used|channel_code|content_type|template_id|
+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
| 132141|         0|Obstetrics & Gyna...|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131911|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131645|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131064|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
| 131752|         0|         Paediatrics|    0_2| 0.01|personalize|         sms|           2|     5679.0|
+-------+----------+--------------------+-------+-----+-----------+------------+------------+-----------+
only showing top 5 rows
+-------+----------+--

## 8. Deduplication & Missing Value Handling

PySpark provides dedicated methods under `.na` for missing value imputation and `.dropDuplicates()` for removing duplicates.

In [ ]:
# Create a sample DataFrame with missing (Null) values and duplicate records
dff = spark.createDataFrame(
    [
        (1, "Shridhar", 25, 50000, "AI"),
        (2, "Amit", None, 65000, "Data Science"),
        (3, "Neha", 28, None, "AI"),
        (4, None, 35, 70000, "MLOps"),
        (5, "Priya", 27, 60000, None),
        (6, "Karan", None, None, "AI"),
        (7, "Sneha", 29, 62000, "Data Science"),
        (8, None, 31, 68000, None),
        (8, None, 31, 68000, None) # Duplicate row
    ],
    ["id", "name", "age", "salary", "department"]
)

print("Original Dataset with Nulls & Duplicates:")
dff.show()

Original Dataset with Nulls & Duplicates:
+---+--------+----+------+------------+
| id|    name| age|salary|  department|
+---+--------+----+------+------------+
|  1|Shridhar|  25| 50000|          AI|
|  2|    Amit|NULL| 65000|Data Science|
|  3|    Neha|  28|  NULL|          AI|
|  4|    NULL|  35| 70000|       MLOps|
|  5|   Priya|  27| 60000|        NULL|
|  6|   Karan|NULL|  NULL|          AI|
|  7|   Sneha|  29| 62000|Data Science|
|  8|    NULL|  31| 68000|        NULL|
|  8|    NULL|  31| 68000|        NULL|
+---+--------+----+------+------------+



In [ ]:
# Remove duplicate rows
print("After removing full row duplicates:")
dff.dropDuplicates().show()

print("After deduplicating based on subset ['id']:")
dff.dropDuplicates(["id"]).show()

After removing full row duplicates:
+---+--------+----+------+------------+
| id|    name| age|salary|  department|
+---+--------+----+------+------------+
|  1|Shridhar|  25| 50000|          AI|
|  7|   Sneha|  29| 62000|Data Science|
|  2|    Amit|NULL| 65000|Data Science|
|  3|    Neha|  28|  NULL|          AI|
|  4|    NULL|  35| 70000|       MLOps|
|  5|   Priya|  27| 60000|        NULL|
|  6|   Karan|NULL|  NULL|          AI|
|  8|    NULL|  31| 68000|        NULL|
+---+--------+----+------+------------+

After deduplicating based on subset ['id']:
+---+--------+----+------+------------+
| id|    name| age|salary|  department|
+---+--------+----+------+------------+
|  1|Shridhar|  25| 50000|          AI|
|  2|    Amit|NULL| 65000|Data Science|
|  3|    Neha|  28|  NULL|          AI|
|  4|    NULL|  35| 70000|       MLOps|
|  5|   Priya|  27| 60000|        NULL|
|  6|   Karan|NULL|  NULL|          AI|
|  7|   Sneha|  29| 62000|Data Science|
|  8|    NULL|  31| 68000|        NULL|

In [ ]:
# Handling Null values using fill()
print("Fill numerical nulls with 0:")
dff.na.fill(0).show()

print("Fill string nulls with 'Unknown':")
dff.na.fill("Unknown").show()

print("Fill missing values using column-specific dictionary:")
dff.na.fill({
    "age": 0,
    "salary": 10000,
    "department": "Unassigned",
    "name": "N/A"
}).show()

print("Fill missing values for specific subset columns:")
dff.na.fill(0, subset=["age", "salary"]).show()

print("Drop rows with ANY null values:")
dff.na.drop().show()

Fill numerical nulls with 0:
+---+--------+---+------+------------+
| id|    name|age|salary|  department|
+---+--------+---+------+------------+
|  1|Shridhar| 25| 50000|          AI|
|  2|    Amit|  0| 65000|Data Science|
|  3|    Neha| 28|     0|          AI|
|  4|    NULL| 35| 70000|       MLOps|
|  5|   Priya| 27| 60000|        NULL|
|  6|   Karan|  0|     0|          AI|
|  7|   Sneha| 29| 62000|Data Science|
|  8|    NULL| 31| 68000|        NULL|
|  8|    NULL| 31| 68000|        NULL|
+---+--------+---+------+------------+

Fill string nulls with 'Unknown':
+---+--------+----+------+------------+
| id|    name| age|salary|  department|
+---+--------+----+------+------------+
|  1|Shridhar|  25| 50000|          AI|
|  2|    Amit|NULL| 65000|Data Science|
|  3|    Neha|  28|  NULL|          AI|
|  4| Unknown|  35| 70000|       MLOps|
|  5|   Priya|  27| 60000|     Unknown|
|  6|   Karan|NULL|  NULL|          AI|
|  7|   Sneha|  29| 62000|Data Science|
|  8| Unknown|  31| 68000|   

## 9. GroupBy & Aggregation Functions

Perform aggregations over groupings using `.groupBy()` combined with `.agg()`.

In [ ]:
# Clean sample DataFrame for group operations
dff_clean = dff.na.drop(subset=["department", "salary"])

# Basic group by department
dff_clean.groupBy("department").avg("salary").show()
dff_clean.groupBy("department").max("salary").show()
dff_clean.groupBy("department").count().show()

+------------+-----------+
|  department|avg(salary)|
+------------+-----------+
|          AI|    50000.0|
|Data Science|    63500.0|
|       MLOps|    70000.0|
+------------+-----------+

+------------+-----------+
|  department|max(salary)|
+------------+-----------+
|          AI|      50000|
|Data Science|      65000|
|       MLOps|      70000|
+------------+-----------+

+------------+-----+
|  department|count|
+------------+-----+
|          AI|    1|
|Data Science|    2|
|       MLOps|    1|
+------------+-----+



In [ ]:
# Multiple aggregations using .agg()
dff_clean.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary"),
    count("*").alias("employee_count")
).show()

+------------+----------+----------+----------+--------------+
|  department|avg_salary|max_salary|min_salary|employee_count|
+------------+----------+----------+----------+--------------+
|          AI|   50000.0|     50000|     50000|             1|
|Data Science|   63500.0|     65000|     62000|             2|
|       MLOps|   70000.0|     70000|     70000|             1|
+------------+----------+----------+----------+--------------+



## 10. DataFrame Joins

PySpark supports standard SQL join types: `inner`, `left`, `right`, `outer`.

In [ ]:
# Create sample tables for join demonstration
emp_df = spark.createDataFrame(
    [(1, "Shridhar", 101), (2, "Amit", 102), (3, "Neha", 101), (4, "Vikram", 103)],
    ["emp_id", "emp_name", "dept_id"]
)

dept_df = spark.createDataFrame(
    [(101, "AI"), (102, "Data Science"), (104, "Finance")],
    ["dept_id", "dept_name"]
)

print("Employee DataFrame:")
emp_df.show()
print("Department DataFrame:")
dept_df.show()

Employee DataFrame:
+------+--------+-------+
|emp_id|emp_name|dept_id|
+------+--------+-------+
|     1|Shridhar|    101|
|     2|    Amit|    102|
|     3|    Neha|    101|
|     4|  Vikram|    103|
+------+--------+-------+

Department DataFrame:
+-------+------------+
|dept_id|   dept_name|
+-------+------------+
|    101|          AI|
|    102|Data Science|
|    104|     Finance|
+-------+------------+



In [ ]:
# Inner Join
print("Inner Join:")
emp_df.join(dept_df, on="dept_id", how="inner").show()

# Left Outer Join
print("Left Join:")
emp_df.join(dept_df, on="dept_id", how="left").show()

# Right Outer Join
print("Right Join:")
emp_df.join(dept_df, on="dept_id", how="right").show()

# Full Outer Join
print("Full Outer Join:")
emp_df.join(dept_df, on="dept_id", how="outer").show()

Inner Join:
+-------+------+--------+------------+
|dept_id|emp_id|emp_name|   dept_name|
+-------+------+--------+------------+
|    101|     1|Shridhar|          AI|
|    102|     2|    Amit|Data Science|
|    101|     3|    Neha|          AI|
+-------+------+--------+------------+

Left Join:
+-------+------+--------+------------+
|dept_id|emp_id|emp_name|   dept_name|
+-------+------+--------+------------+
|    101|     1|Shridhar|          AI|
|    102|     2|    Amit|Data Science|
|    101|     3|    Neha|          AI|
|    103|     4|  Vikram|        NULL|
+-------+------+--------+------------+

Right Join:
+-------+------+--------+------------+
|dept_id|emp_id|emp_name|   dept_name|
+-------+------+--------+------------+
|    101|     3|    Neha|          AI|
|    101|     1|Shridhar|          AI|
|    102|     2|    Amit|Data Science|
|    104|  NULL|    NULL|     Finance|
+-------+------+--------+------------+

Full Outer Join:
+-------+------+--------+------------+
|dept_id|

## 11. Spark SQL Engine Execution

Register a DataFrame as a Temporary View to run native ANSI SQL queries.

In [ ]:
# Register temporary view
df.createOrReplaceTempView("campaign_data")

# Run Spark SQL query
sql_result = spark.sql("""
    SELECT user_specialty, COUNT(*) as campaign_count, AVG(score) as avg_score
    FROM campaign_data
    WHERE score > 0.005
    GROUP BY user_specialty
    ORDER BY avg_score DESC
""")

sql_result.show(5)

+------------------+--------------+--------------------+
|    user_specialty|campaign_count|           avg_score|
+------------------+--------------+--------------------+
|  General Medicine|          3014|0.010000000000000635|
|   General Surgery|           174|0.010000000000000007|
|Community Medicine|           149|0.010000000000000007|
|       Dermatology|           111|0.010000000000000007|
|         Pathology|           155|0.010000000000000007|
+------------------+--------------+--------------------+
only showing top 5 rows


## 12. User Defined Functions (UDFs)

UDFs allow custom Python functions to be applied across DataFrame columns.

In [ ]:
# Define custom Python function
def categorize_score(score):
    if score is None:
        return "Unknown"
    elif score >= 0.01:
        return "High Impact"
    elif score >= 0.005:
        return "Medium Impact"
    else:
        return "Low Impact"

# Register UDF with explicit return type
score_udf = udf(categorize_score, StringType())

# Apply UDF to DataFrame
df_udf = df.withColumn("impact_level", score_udf(col("score")))
df_udf.select("user_id", "score", "impact_level").show(5)

+-------+-----+------------+
|user_id|score|impact_level|
+-------+-----+------------+
| 131060| 0.01| High Impact|
| 131064| 0.01| High Impact|
| 131065| 0.01| High Impact|
| 131077| 0.01| High Impact|
| 131098| 0.01| High Impact|
+-------+-----+------------+
only showing top 5 rows


## 13. Date & Timestamp Manipulations

PySpark provides date functions to extract year, month, day, calculate date differences, and format timestamps.

In [ ]:
# Create sample DataFrame with dates
date_df = spark.createDataFrame(
    [
        (1, "2024-01-15 10:30:00"),
        (2, "2024-06-20 14:45:00"),
        (3, "2024-11-05 09:15:00")
    ],
    ["id", "timestamp_str"]
)

# Parse string to Timestamp type
date_df = date_df.withColumn("event_time", to_timestamp(col("timestamp_str")))

# Extract Date components
date_df = date_df \
    .withColumn("year", year(col("event_time"))) \
    .withColumn("month", month(col("event_time"))) \
    .withColumn("day", dayofmonth(col("event_time"))) \
    .withColumn("current_date", current_date()) \
    .withColumn("future_date", date_add(col("event_time"), 10)) \
    .withColumn("days_diff", datediff(current_date(), col("event_time")))

date_df.show()

+---+-------------------+-------------------+----+-----+---+------------+-----------+---------+
| id|      timestamp_str|         event_time|year|month|day|current_date|future_date|days_diff|
+---+-------------------+-------------------+----+-----+---+------------+-----------+---------+
|  1|2024-01-15 10:30:00|2024-01-15 10:30:00|2024|    1| 15|  2026-09-16| 2024-01-25|      975|
|  2|2024-06-20 14:45:00|2024-06-20 14:45:00|2024|    6| 20|  2026-09-16| 2024-06-30|      818|
|  3|2024-11-05 09:15:00|2024-11-05 09:15:00|2024|   11|  5|  2026-09-16| 2024-11-15|      680|
+---+-------------------+-------------------+----+-----+---+------------+-----------+---------+



## 14. String Functions

PySpark built-in string transformation functions: `upper()`, `lower()`, `length()`, `concat()`, `substring()`, `trim()`.

In [ ]:
# Apply string functions
df_strings = df.select(
    col("user_specialty"),
    upper(col("user_specialty")).alias("specialty_uppercase"),
    lower(col("user_specialty")).alias("specialty_lowercase"),
    length(col("user_specialty")).alias("specialty_length"),
    concat(col("user_specialty"), lit(" - Specialization")).alias("concatenated_str"),
    substring(col("user_specialty"), 1, 5).alias("short_code")
)

df_strings.show(5, truncate=False)

+------------------------+------------------------+------------------------+----------------+-----------------------------------------+----------+
|user_specialty          |specialty_uppercase     |specialty_lowercase     |specialty_length|concatenated_str                         |short_code|
+------------------------+------------------------+------------------------+----------------+-----------------------------------------+----------+
|Obstetrics & Gynaecology|OBSTETRICS & GYNAECOLOGY|obstetrics & gynaecology|24              |Obstetrics & Gynaecology - Specialization|Obste     |
|Paediatrics             |PAEDIATRICS             |paediatrics             |11              |Paediatrics - Specialization             |Paedi     |
|Internal Medicine       |INTERNAL MEDICINE       |internal medicine       |17              |Internal Medicine - Specialization       |Inter     |
|Paediatrics             |PAEDIATRICS             |paediatrics             |11              |Paediatrics - Specializat

## 15. Advanced Window Functions

Window functions perform calculations across a set of rows related to the current row.

### Key Functions:
- **Ranking**: `row_number()`, `rank()`, `dense_rank()`
- **Analytical**: `lag()`, `lead()`
- **Aggregations**: `sum()`, `avg()` over window specification

In [ ]:
# Define sample data for Window Functions
salaries_df = spark.createDataFrame(
    [
        ("AI", "Shridhar", 85000),
        ("AI", "Neha", 90000),
        ("AI", "Karan", 75000),
        ("Data Science", "Amit", 65000),
        ("Data Science", "Sneha", 72000),
        ("Data Science", "Priya", 72000),
        ("MLOps", "Vikram", 70000)
    ],
    ["department", "name", "salary"]
)

# 1. Define Window Partitioned by Department and Ordered by Salary Descending
window_spec_dept = Window.partitionBy("department").orderBy(desc("salary"))

# 2. Define Window for Lag/Lead across entire dataset ordered by Salary
window_spec_global = Window.orderBy("salary")

# Apply Window Functions
window_df = salaries_df \
    .withColumn("row_num", row_number().over(window_spec_dept)) \
    .withColumn("rank", rank().over(window_spec_dept)) \
    .withColumn("dense_rank", dense_rank().over(window_spec_dept)) \
    .withColumn("prev_salary", lag("salary", 1).over(window_spec_global)) \
    .withColumn("next_salary", lead("salary", 1).over(window_spec_global))

window_df.show()

+------------+--------+------+-------+----+----------+-----------+-----------+
|  department|    name|salary|row_num|rank|dense_rank|prev_salary|next_salary|
+------------+--------+------+-------+----+----------+-----------+-----------+
|          AI|    Neha| 90000|      1|   1|         1|      85000|       NULL|
|          AI|Shridhar| 85000|      2|   2|         2|      75000|      90000|
|          AI|   Karan| 75000|      3|   3|         3|      72000|      85000|
|Data Science|   Sneha| 72000|      1|   1|         1|      70000|      72000|
|Data Science|   Priya| 72000|      2|   1|         1|      72000|      75000|
|Data Science|    Amit| 65000|      3|   3|         2|       NULL|      70000|
|       MLOps|  Vikram| 70000|      1|   1|         1|      65000|      72000|
+------------+--------+------+-------+----+----------+-----------+-----------+



## 16. Pandas Interoperability & Data Exporting

Convert PySpark DataFrames to Pandas or export to standard storage formats (CSV, Parquet, JSON).

In [ ]:
# Convert PySpark DataFrame to Pandas DataFrame
pdf = df.limit(5).toPandas()
print("Pandas DataFrame type:", type(pdf))
pdf

Pandas DataFrame type: <class 'pandas.core.frame.DataFrame'>


In [ ]:
# Exporting DataFrames (Uncomment to execute write operations)
# df.write.mode("overwrite").option("header", "true").csv("output_csv")
# df.write.mode("overwrite").parquet("output_parquet")
# df.write.mode("append").json("output_json")
print("Write syntax demo ready!")

Write syntax demo ready!


## 17. PySpark Quick Reference Cheat Sheet 🧠

| Category | Operation | PySpark Code Example |
| :--- | :--- | :--- |
| **Initialization** | Create Session | `spark = SparkSession.builder.appName("App").getOrCreate()` |
| **Read Data** | Read CSV | `spark.read.csv("file.csv", header=True, inferSchema=True)` |
| **Read Data** | Read Parquet | `spark.read.parquet("file.parquet")` |
| **Inspection** | View Rows | `df.show(5, truncate=False)` |
| **Inspection** | View Schema | `df.printSchema()` |
| **Inspection** | Row Count | `df.count()` |
| **Selection** | Select & Alias | `df.select(col("A").alias("B"))` |
| **Filtering** | Multiple Filters | `df.filter((col("A") > 10) & (col("B") == "X"))` |
| **Columns** | Add Column | `df.withColumn("new_col", col("A") * 2)` |
| **Columns** | Conditional Column | `df.withColumn("flag", when(col("A") > 5, 1).otherwise(0))` |
| **Columns** | Drop Column | `df.drop("col_name")` |
| **Null Values** | Fill Nulls | `df.na.fill({"colA": 0, "colB": "N/A"})` |
| **Null Values** | Drop Nulls | `df.na.drop(subset=["colA"])` |
| **GroupBy** | Aggregations | `df.groupBy("dept").agg(avg("salary"), count("*"))` |
| **Joins** | DataFrame Join | `df1.join(df2, on="id", how="left")` |
| **SQL** | Temp View | `df.createOrReplaceTempView("table_name")` |
| **Window** | Rank Over Window | `rank().over(Window.partitionBy("dept").orderBy(desc("salary")))` |
| **Export** | Write Parquet | `df.write.mode("overwrite").parquet("path")` |

---
🎉 **End of PySpark Tutorial & Cheat Sheet! Happy Coding!**